# QuantVault exploration

Run from the repository root (`jupyter lab`). Sections: 1) risk assessment and Mosca, 2) one BB84 run, 3) noise sweep.
Outputs are cleared before commit; the inventory is illustrative sample data.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))
import pandas as pd
from src.risk import RiskConfig, analyse_inventory

## 1. Risk assessment: change the threat horizon Z

In [ ]:
Z = 15  # assumed years until a cryptographically relevant quantum computer (an assumption, not a prediction)
res = analyse_inventory(cfg=RiskConfig(threat_horizon_years=Z))
pd.DataFrame([{"rank": r["rank"], "asset": r["name"], "tier": r["tier"], "R": r["risk_score"],
               "exposure": r["security_exposure"], "mosca": r["mosca"]["status"], "margin_yrs": r["mosca"]["margin"]}
              for r in res])

## 2. One BB84 run on Qiskit Aer

In [ ]:
from src.bb84 import BB84Config
from src.postprocessing import run_qkd
for eve in (False, True):
    r = run_qkd(BB84Config(n_qubits=2048, eve_enabled=eve, seed=1))
    print(f"Eve={eve}: sifted={r.n_sifted} QBER={r.qber_sample:.1%} upper={r.qber_upper:.1%} "
          f"final_key={r.final_length} accepted={r.accepted} {r.abort_reason}")

## 3. Noise sweep (experiments A-D)

In [ ]:
from src.experiments import run_grid, aggregate
grid = run_grid(n_qubits=1024, reps=3, noise_models=("depolarizing",))
aggregate(grid, ["experiment", "noise_p", "eve"])[["experiment", "noise_p", "eve", "qber_mean", "accept_rate", "final_len_mean"]]